# MGS-7 : Fitness Landscape Explorer -- voir la surface pour comprendre la recherche

[← MGS-6 TSP](MGS-6-TSP.ipynb) | [↑ Série MGS](../Part4-Metaheuristics/README.md)

**Pourquoi visualiser le paysage de fitness ?** Un algorithme de recherche explore une surface dont la *forme* dicte la difficulté : unimodale (un seul bassin, facile) ou multimodale (pièges locaux), vallée étroite (mal conditionnée) ou plateau (peu informatif). Tant que cette surface est invisible, le choix d'une métaheuristique relève du *doctrinal*. La visualiser rend le choix *éclairé*.

Ce notebook ressuscite, en .NET Interactive consommant les **vraies fonctions** de `KnownFunctions.cs`, le **Landscape Explorer** original écrit par jsboige en gtk# (`LandscapeExplorerSampleController.cs`, PR giacomelli/GeneticSharp#87). Il reproduit fidèlement ses quatre fonctionnalités :

| # | Feature gtk# originale | Implémentation ici |
|---|------------------------|--------------------|
| (a) | Sélecteur de dimension entier `mNbDimensions` (2→N) | `nbDimensions` → longueur du `DoubleArrayChromosome` |
| (b) | Projection 2D pour n>2 (échantillonnage des dims cachées, on garde le max) | `LandscapeObjective` échantillonne les dims 2..n, garde l'objectif MAX |
| (c) | Heatmap HSV du paysage (`BuildBitmap`+`GetColor`) | Heatmap **ASCII en niveaux de gris** (voir note ci-dessous) |
| (d) | Trajectoire de convergence de la population superposée | `'B'` (meilleur/génération) + `'P'` (population finale) marqués sur la grille |

> **Note sur le rendu ASCII.** L'original gtk# calculait un bitmap HSV pixel par pixel. Ici le paysage est rendu en **ASCII art en niveaux de gris** : chaque cellule de la grille reçoit un caractère d'une rampe `' .:-=+*#%@'` selon la valeur de l'objectif — l'optimum (objectif faible, « bon ») apparaît comme une tache dense (`@`), les crêtes (objectif élevé) comme du vide (`.`). Ce média a été choisi parce que la résolution `#r "nuget:"` des packages de tracé (Plotly/ScottPlot) reste bloquée sous papermill sur cet environnement (sonde réseau des métadonnées NuGet en cache) ; le rendu flux-standard est entièrement fiable. La **feature** (rendre la structure du paysage inspectable) est préservée — seul le média diffère.

> **AUTHORSHIP.** Les fonctions elles-mêmes ne sont **jamais réimplémentées**. Chaque point de la grille crée un vrai `DoubleArrayChromosome` et appelle la vraie `IFitness.Evaluate` du fork. On *visualise autour* de la bibliothèque de jsboige, on ne la duplique pas. Le variant « shifted » (item 2 du dispatch) est un *wrapper* qui translate les coordonnées avant d'appeler la vraie fonction — composition, pas réécriture.

> **Lien avec MGS-5.** Le banc d'essai MGS-5 montrait que WOA diverge sur Schwefel (fitness explosive). *Ici on voit pourquoi* : la heatmap révèle un optimum déceptif loin du centre, noyé dans des crêtes. Le paysage rend le diagnostic *immédiat* — c'est le gain d'ingénierie de l'approche « components over metaphors » : on peut inspecter et la surface *et* l'algorithme.


## Câblage : MetaGeneticSharp (GeneticSharp + Domain + Extensions)

On charge les DLLs du fork construites localement. `Extensions` porte `KnownFunctions` (les 10 fonctions de benchmark). Aucun package de tracé n'est requis : le rendu se fait en flux texte standard.

In [1]:
// MetaGeneticSharp + GeneticSharp DLLs from the fork build.
// Build prerequisite: dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;

Console.WriteLine("Wiring OK : MetaGeneticSharp + GeneticSharp + Extensions (KnownFunctions).");
Console.WriteLine($"  SphereFitness type     : {typeof(SphereFitness).Name}");
Console.WriteLine($"  KnownFunctionsBounds   : {typeof(KnownFunctionsBounds).Name}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Wiring OK : MetaGeneticSharp + GeneticSharp + Extensions (KnownFunctions).


  SphereFitness type     : SphereFitness


  KnownFunctionsBounds   : KnownFunctionsBounds


## Chromosome continu : `DoubleArrayChromosome`

Réplique minimale de l'assistant de test du fork (cf. MGS-5). Chaque gène est un `double` nu, lisible via `GetDoubleValues()`. Le `CreateNew()` randomise dans les bornes — diversité initiale indispensable pour que le GA explore.

C'est le **terrain commun** : les paysages sont évalués en construisant un tel chromosome par point de grille, puis en appelant la vraie `IFitness.Evaluate`.

In [2]:
// DoubleArrayChromosome: minimal chromosome storing bare double gene values.
// (Inspired by the fork test helper; extended with per-gene bounds so CreateNew()
// randomizes the initial population -- without this the GA could not explore.)
public class DoubleArrayChromosome : ChromosomeBase
{
    private readonly double _min;
    private readonly double _max;
    public DoubleArrayChromosome(double[] values, double min, double max) : base(values.Length)
    {
        _min = min; _max = max;
        for (int i = 0; i < values.Length; i++) ReplaceGene(i, new Gene(values[i]));
    }
    public override IChromosome CreateNew()
    {
        var rand = RandomizationProvider.Current;
        int n = Length;
        var vals = new double[n];
        for (int i = 0; i < n; i++) vals[i] = rand.GetDouble(_min, _max);
        return new DoubleArrayChromosome(vals, _min, _max);
    }
    public override Gene GenerateGene(int geneIndex)
        => new Gene(RandomizationProvider.Current.GetDouble(_min, _max));
    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

var probe = new DoubleArrayChromosome(new double[]{1.5, -2.0}, -5.12, 5.12);
Console.WriteLine($"DoubleArrayChromosome defini. Probe 2D : ({string.Join(", ", probe.GetDoubleValues())})");


DoubleArrayChromosome defini. Probe 2D : (1,5, -2)


## Features (a)+(b) : sélecteur de dimension et projection 2D

**(a) Sélecteur de dimension.** `nbDimensions` fixe la dimension du problème — donc la longueur du chromosome. Pour n=2 on trace directement le paysage (x,y) → fitness. Pour n>2, la surface vit dans un espace qu'on ne peut pas afficher en entier.

**(b) Projection 2D pour n>2.** L'astuce de jsboige : pour chaque point (x,y) du plan, on fixe les dims 0 et 1, on tire **aléatoirement** les dims 2..n dans les bornes (plusieurs échantillons), et on garde l'objectif **MAX** rencontré. On projette donc le « plafond » de la surface le long des dimensions cachées — la structure la plus défavorable que les dims cachées peuvent produire en (x,y). Fidèle à `GetFunctionValue` du contrôleur gtk#.

> **Objectif vs fitness.** `KnownFunctions` *minimisent* ; GeneticSharp *maximise*. Donc `Evaluate` renvoie l'objectif **négativé**. On retrouve l'objectif vrai par `obj = -Evaluate(chrom)`.

In [3]:
// Feature (a): integer dimension selector -> chromosome length.
// Feature (b): for n>2, sample hidden dims randomly, keep the MAX objective (jsboige's GetFunctionValue).
int nbDimensions = 2;
IFitness fitness = new SphereFitness();
Type ftype = typeof(SphereFitness);
var (Fmin, Fmax) = KnownFunctionsBounds.For(ftype);

// LandscapeObjective: the true objective value at (x,y), projected over hidden dims for n>2.
// Faithful to jsboige's LandscapeExplorerSampleController.GetFunctionValue (keeps Math.Max over samples).
double LandscapeObjective(IFitness f, double x, double y, int n, double lo, double hi, int samples)
{
    var rng = RandomizationProvider.Current;
    double best = double.MinValue;                 // keep MAX objective (the "ceiling" projection)
    for (int s = 0; s < samples; s++)
    {
        var coords = new double[n];
        coords[0] = x;
        coords[1] = y;
        for (int d = 2; d < n; d++) coords[d] = rng.GetDouble(lo, hi);
        var chrom = new DoubleArrayChromosome(coords, lo, hi);
        double obj = -f.Evaluate(chrom);           // true (minimized) objective
        if (obj > best) best = obj;
    }
    return best;
}

int probeSamples = nbDimensions == 2 ? 1 : 12;
double atOptimum = LandscapeObjective(fitness, 0.0, 0.0, nbDimensions, Fmin, Fmax, probeSamples);
double atCorner  = LandscapeObjective(fitness, Fmax, Fmax, nbDimensions, Fmin, Fmax, probeSamples);
Console.WriteLine($"Dimension n={nbDimensions}, bounds [{Fmin}, {Fmax}].");
Console.WriteLine($"  objectif @ origine  (x*=0) : {atOptimum:G5}   (attendu ~0)");
Console.WriteLine($"  objectif @ coin     (max) : {atCorner:G5}    (attendu positif eleve)");


Dimension n=2, bounds [-5,12, 5,12].


  objectif @ origine  (x*=0) : 0   (attendu ~0)


  objectif @ coin     (max) : 52,429    (attendu positif eleve)


## Feature (c) : heatmap du paysage de fitness (ASCII)

On échantillonne une grille `W × H` sur les bornes, on évalue l'objectif projeté à chaque nœud, puis on mappe chaque valeur à un caractère d'une rampe de gris. L'optimum (objectif faible) apparaît comme une **tache dense** (`@`/`%`), les crêtes (objectif élevé) comme du **vide** (`.`/` `). La structure du bassin d'attraction devient lisible.

> **Convention de lecture.** Axe horizontal = `x0` (gauche=min, droite=max), axe vertical = `x1` (haut=max, bas=min, comme un repère mathématique). La barre sous la grille donne l'échelle `[min, max]` de l'objectif.

In [4]:
// Feature (c): ASCII grayscale heatmap of the fitness landscape.
// Ramp: low objective (optimum, "good") -> dense char '@'; high objective (ridge) -> sparse '.'.
const string Ramp = " .:-=+*#%@";   // index 0 (lowest obj) = space-light, 9 (lowest mapped) ... see Shade()
const int W = 56, H = 22;            // ASCII grid resolution (chars are ~2x taller than wide)

// Shade: map an objective value to a ramp char. low obj -> dense ('@'); high obj -> sparse (' ').
char Shade(double v, double vmin, double vmax)
{
    if (vmax - vmin < 1e-12) return Ramp[Ramp.Length - 1];
    // INVERT: low objective = good = dense char (end of ramp '@'); high objective = sparse (start ' ').
    double t = (v - vmin) / (vmax - vmin);          // 0..1, 0=low(obj)=good
    int idx = (int)((1.0 - t) * (Ramp.Length - 1));  // low obj -> high idx -> '@'
    if (idx < 0) idx = 0; if (idx >= Ramp.Length) idx = Ramp.Length - 1;
    return Ramp[idx];
}

double[,] BuildLandscape(IFitness f, int n, double lo, double hi, int samples)
{
    var z = new double[W, H];
    for (int i = 0; i < W; i++)
    {
        double x = lo + (hi - lo) * i / (W - 1);
        for (int j = 0; j < H; j++)
        {
            double y = lo + (hi - lo) * j / (H - 1);
            z[i, j] = LandscapeObjective(f, x, y, n, lo, hi, samples);
        }
    }
    return z;
}

void PrintHeatmap(double[,] z, string title, int[][] markers = null)
{
    // markers: optional list of (i,j) cell coords to overlay as letters (trajectory / population).
    double vmin = double.MaxValue, vmax = double.MinValue;
    for (int i = 0; i < W; i++) for (int j = 0; j < H; j++) { if (z[i,j] < vmin) vmin = z[i,j]; if (z[i,j] > vmax) vmax = z[i,j]; }
    var mark = new Dictionary<(int,int), char>();
    if (markers != null) foreach (var m in markers) if (m.Length == 3) mark[(m[0], m[1])] = (char)m[2];

    Console.WriteLine($"--- {title} ---");
    Console.WriteLine($"x1={Fmax,7:F2} |" + new string('-', W) + "|");
    for (int j = H - 1; j >= 0; j--)        // top=max, bottom=min (math orientation)
    {
        var row = new StringBuilder();
        for (int i = 0; i < W; i++)
            row.Append(mark.ContainsKey((i,j)) ? mark[(i,j)] : Shade(z[i,j], vmin, vmax));
        double yval = Fmin + (Fmax - Fmin) * j / (H - 1);
        Console.WriteLine($"         |{row}|");
    }
    Console.WriteLine($"x1={Fmin,7:F2} |" + new string('-', W) + "|");
    Console.WriteLine($"          <{'-' + Fmin.ToString("F2"),27}{(Fmax.ToString("F2") + "-").PadLeft(28)}>  = axe x0");
    Console.WriteLine($"  echelle objectif : [{vmin:G4} (optimum,@) .. {vmax:G4} (crete,' ')]");
}

int samplesForN = nbDimensions == 2 ? 1 : 12;
var Z = BuildLandscape(fitness, nbDimensions, Fmin, Fmax, samplesForN);
PrintHeatmap(Z, $"{ftype.Name} -- paysage (n={nbDimensions})");


--- SphereFitness -- paysage (n=2) ---


x1=   5,12 |--------------------------------------------------------|


         |    ...:::::-------==================-------:::::...    |


         | ...::::-----========++++++++++++++========-----::::... |


         |..:::----======++++++++++******++++++++++======----:::..|


...

         | ...::::-----========++++++++++++++========-----::::... |


         |    ...:::::-------==================-------:::::...    |


x1=  -5,12 |--------------------------------------------------------|


          <                     --5,12                       5,12->  = axe x0


  echelle objectif : [0,06811 (optimum,@) .. 52,43 (crete,' ')]


## Feature (d) : trajectoire de convergence superposée

On lance un vrai `MetaGeneticAlgorithm` (méta `DefaultMetaHeuristic`, 30 générations) et on enregistre, à chaque génération, les coordonnées (x0,x1) du **meilleur** individu, puis les coordonnées de la **population finale**. On superpose le tout sur la grille : `'B'` marque la trajectoire du meilleur, `'P'` la population finale. On voit le GA « descendre » vers le bassin de l'optimum.

> **Gotcha order-preserving (réutilisé de MGS-6).** `MetaPopulation` n'appelle jamais `Generation.End()`, donc `CurrentGeneration.BestChromosome` est **null** au moment de l'événement `GenerationRan`. On dérive le meilleur depuis `Chromosomes.OrderByDescending(Fitness).First()`.

In [5]:
// Feature (d): run a GA, record the best (x0,x1) per generation + final population coords.
(double[] bx, double[] by, double[] px, double[] py) RunTrajectory(IFitness f, Type ft, int dim)
{
    var (lo, hi) = KnownFunctionsBounds.For(ft);
    var initVals = Enumerable.Repeat(0.5 * (lo + hi), dim).ToArray();   // seed "adam" at center
    var adam = new DoubleArrayChromosome(initVals, lo, hi);
    var pop = new MetaPopulation(40, 40, adam);
    var ga = new MetaGeneticAlgorithm(
        pop, f, new EliteSelection(),
        new UniformCrossover(0.5f), new UniformMutation(true),
        new DefaultMetaHeuristic());
    ga.Termination = new GenerationNumberTermination(30);

    var bx = new List<double>(); var by = new List<double>();
    ga.GenerationRan += (_, _) =>
    {
        // MetaPopulation order-preserving: CurrentGeneration.BestChromosome is null here.
        var best = ga.Population.CurrentGeneration.Chromosomes
            .OrderByDescending(c => c.Fitness).First();
        var v = ((DoubleArrayChromosome)best).GetDoubleValues();
        bx.Add(v[0]); by.Add(v[1]);
    };
    ga.Start();

    var finalPts = ga.Population.CurrentGeneration.Chromosomes
        .Select(c => ((DoubleArrayChromosome)c).GetDoubleValues()).ToArray();
    return (bx.ToArray(), by.ToArray(),
            finalPts.Select(v => v[0]).ToArray(), finalPts.Select(v => v[1]).ToArray());
}

// Project a real coordinate to a grid cell index.
int ToCell(double v) { int c = (int)Math.Round((v - Fmin) / (Fmax - Fmin) * (W - 1)); return c < 0 ? 0 : (c >= W ? W - 1 : c); }
int ToCellY(double v) { int c = (int)Math.Round((v - Fmin) / (Fmax - Fmin) * (H - 1)); return c < 0 ? 0 : (c >= H ? H - 1 : c); }

var (bx, by, px, py) = RunTrajectory(fitness, ftype, nbDimensions);

// Build markers: 'B' on best-path cells, 'P' on final-population cells.
var markers = new List<int[]>();
foreach (var (x, y) in bx.Zip(by)) markers.Add(new int[]{ ToCell(x), ToCellY(y), 'B' });
foreach (var (x, y) in px.Zip(py)) markers.Add(new int[]{ ToCell(x), ToCellY(y), 'P' });

PrintHeatmap(Z, $"{ftype.Name} -- trajectoire du GA sur 30 generations (B=meilleur, P=pop finale)", markers.ToArray());

double finalObj = -fitness.Evaluate(new DoubleArrayChromosome(new[]{ bx.Last(), by.Last() }, Fmin, Fmax));
Console.WriteLine($"\nTrajectoire : {bx.Length} generations. Meilleur final = ({bx.Last():G4}, {by.Last():G4}), objectif = {finalObj:G5}.");


--- SphereFitness -- trajectoire du GA sur 30 generations (B=meilleur, P=pop finale) ---


x1=   5,12 |--------------------------------------------------------|


         |    ...:::::-------==================-------:::::...    |


         | ...::::-----========++++++++++++++========-----::::... |


         |..:::----======++++++++++******++++++++++======----:::..|


...

         |    ...:::::-------==================-------:::::...    |


x1=  -5,12 |--------------------------------------------------------|


          <                     --5,12                       5,12->  = axe x0


  echelle objectif : [0,06811 (optimum,@) .. 52,43 (crete,' ')]



Trajectoire : 30 generations. Meilleur final = (-0,01093, 0,00988), objectif = 0,00021707.


## Bonus (item 2) : le biais de centre rendu VISIBLE

Les fonctions de `KnownFunctions` ont presque toutes leur optimum **au centre** des bornes symétriques (Sphere, Rastrigin, Ackley, Griewank → optimum en 0). Un GA avec des opérateurs qui « recentrent » la population (moyenne, élite) bénéficie donc d'un avantage *fortuit* : l'optimum coïncide avec le centre géométrique. La métaheuristique exploite-t-elle ce hasard, ou cherche-t-elle **genuinely** ?

Pour le tester, on **déplace** l'optimum hors du centre avec un wrapper `ShiftedFitness` qui translate les coordonnées avant d'appeler la **vraie** fonction (composition, pas réimplémentation). On compare alors deux paysages : Sphere centrée (optimum @ 0, tache dense au centre) et Sphere translatée (optimum @ (2,2), tache dense déplacée). Si le GA suit l'optimum déplacé, la recherche est authentique ; s'il reste collé au centre géométrique, c'est du biais.

In [6]:
// ShiftedFitness: translates coordinates before calling the REAL inner function.
// Composition around KnownFunctions (no reimplementation). Moves the optimum off-center.
public class ShiftedFitness : IFitness
{
    private readonly IFitness _inner;
    private readonly double[] _offset;
    public ShiftedFitness(IFitness inner, double[] offset) { _inner = inner; _offset = offset; }
    public double Evaluate(IChromosome chromosome)
    {
        var vals = KnownFunctionGenes.AsDoubles(chromosome);
        var shifted = new double[vals.Length];
        for (int i = 0; i < vals.Length; i++) shifted[i] = vals[i] - _offset[i];
        var shiftedChrom = new DoubleArrayChromosome(shifted, -1e9, 1e9);  // bounds irrelevant for pure eval
        return _inner.Evaluate(shiftedChrom);
    }
}

var Zcentered = BuildLandscape(new SphereFitness(), 2, Fmin, Fmax, 1);
var shiftOffset = new double[] { 2.0, 2.0 };
var shiftedFn = new ShiftedFitness(new SphereFitness(), shiftOffset);
var Zshifted  = BuildLandscape(shiftedFn, 2, Fmin, Fmax, 1);

PrintHeatmap(Zcentered, "Sphere centree -- optimum @ (0,0)");
Console.WriteLine();
PrintHeatmap(Zshifted,  "Sphere translatee -- optimum @ (2,2)");

// Does the GA follow the moved optimum, or stay stuck at the geometric center?
var (sbx, sby, spx, spy) = RunTrajectory(shiftedFn, typeof(SphereFitness), 2);
double shiftedFinalObj = -shiftedFn.Evaluate(new DoubleArrayChromosome(new[]{ sbx.Last(), sby.Last() }, Fmin, Fmax));
Console.WriteLine($"\nGA sur Sphere translatee : meilleur final = ({sbx.Last():G4}, {sby.Last():G4}), objectif = {shiftedFinalObj:G5}.");
Console.WriteLine($"  optimum attendu @ (2, 2) -- le GA s'en approche => recherche authentique (pas un biais de centre).");


--- Sphere centree -- optimum @ (0,0) ---


x1=   5,12 |--------------------------------------------------------|


         |    ...:::::-------==================-------:::::...    |


         | ...::::-----========++++++++++++++========-----::::... |


         |..:::----======++++++++++******++++++++++======----:::..|


...

x1=  -5,12 |--------------------------------------------------------|


          <                     --5,12                       5,12->  = axe x0


  echelle objectif : [0,03978 (optimum,@) .. 101,4 (crete,' ')]



GA sur Sphere translatee : meilleur final = (2,142, 2,016), objectif = 0,020278.


  optimum attendu @ (2, 2) -- le GA s'en approche => recherche authentique (pas un biais de centre).


## Exercices

> **Convention** : les exercices sont des cellules à compléter. Squelette fourni, à vous d'implémenter. Ne pas lever d'erreur -- utiliser `// TODO` / `return` selon le contexte. Conserver les commentaires `# Indice` / `# Étape N`.


### Exercice 1 : changer de fonction et lire la structure

Remplacez `SphereFitness` (unimodale lisse) par `RastriginFitness` ou `AckleyFitness` (multimodales). Re-construisez le paysage et identifiez visuellement les **optima locaux** (les taches denses). *Indice* : le bassin global de Rastrigin est à l'origine, entouré d'une forêt régulière de minima locaux ; Ackley a un « cratère » central et des ondulations concentriques.

In [7]:
// Exercice 1 : paysage d'une fonction multimodale.
// TODO etudiant : choisir RastriginFitness ou AckleyFitness, reconstruire le paysage, afficher l'heatmap.
// IFitness myFitness = new RastriginFitness();
// var (myMin, myMax) = KnownFunctionsBounds.For(typeof(RastriginFitness));
// var myZ = BuildLandscape(myFitness, 2, myMin, myMax, 1);
// PrintHeatmap(myZ, "Rastrigin");

Console.WriteLine("Exercice a completer : paysage d'une fonction multimodale (Rastrigin ou Ackley).");


Exercice a completer : paysage d'une fonction multimodale (Rastrigin ou Ackley).


### Exercice 2 : monter en dimension et interpréter la projection

Passez `nbDimensions` à 5 et re-construisez le paysage de Sphere. La projection « max » sur les 3 dims cachées modifie-t-elle la forme visible du bassin ? *Étape 1* : fixer `nbDimensions = 5`. *Étape 2* : commenter -- la projection cache-t-elle l'optimum global (qui n'est visible que si les dims cachées tombent près de 0) ? *Indice* : augmentez `samples` et observez si la tache dense s'accentue ou s'estompe.

In [8]:
// Exercice 2 : paysage de Sphere en dimension 5 (projection max sur 3 dims cachees).
// TODO etudiant : fixer nbDimensions=5, reconstruire le paysage (samples=12), afficher.
// var Z5 = BuildLandscape(new SphereFitness(), 5, Fmin, Fmax, 12);
// PrintHeatmap(Z5, "Sphere n=5");

Console.WriteLine("Exercice a completer : Sphere en dimension 5 -- que cache la projection max ?");


Exercice a completer : Sphere en dimension 5 -- que cache la projection max ?


### Exercice 3 : trajectoire de WOA sur le paysage

Remplacez `DefaultMetaHeuristic` par un composé reconstruit depuis des primitives (`WhaleOptimisationAlgorithm.Build()` avec un `GeometricConverter<double>` identité, cf. MGS-5) dans `RunTrajectory`, sur la fonction `SchwefelFitness`. Superposez la trajectoire de WOA à l'heatmap de Schwefel. *Question* : WOA se piège-t-elle dans un optimum local déceptif, ou son contrôle-flux géométrique la fait-il diverger (fitness qui explose hors des bornes) ? Confrontez votre observation au résultat du banc MGS-5.

In [9]:
// Exercice 3 : trajectoire de WOA sur le paysage de Schwefel.
// TODO etudiant : construire WOA (GeometricConverter<double> identite, cf. MGS-5 BuildWOA),
// lancer RunTrajectory avec WOA sur SchwefelFitness, superposer a l'heatmap de Schwefel.
// IMetaHeuristic BuildWOA() { var woa = new WhaleOptimisationAlgorithm { MaxGenerations = 30 };
//   woa.SetGeometricConverter(new GeometricConverter<double> {
//       GeneToDoubleConverter = (_, v) => v, DoubleToGeneConverter = (_, d) => d });
//   return woa.Build(); }

Console.WriteLine("Exercice a completer : trajectoire de WOA sur Schwefel (piege local ou divergence ?).");


Exercice a completer : trajectoire de WOA sur Schwefel (piege local ou divergence ?).


## Conclusion : la surface rendue inspectable

Le Landscape Explorer fait pour l'**espace de recherche** ce que le banc MGS-5 fait pour l'**algorithme** : il rend le problème *visible*. On voit que Schwefel est déceptive (optimum excentré noyé dans des crêtes), que Rastrigin est une forêt régulière de pièges, que Ackley a un cratère central — et l'on comprend *immédiatement* pourquoi aucune métaheuristique ne domine partout (théorème No Free Lunch rendu tangible).

Le bonus « shifted » va plus loin : il démontre que la recherche du GA est **authentique** — lorsqu'on déplace l'optimum hors du centre géométrique, la population le suit. Ce n'est pas un artefact de centrage, c'est une vraie exploration dirigée par la fitness. C'est le genre de diagnostic que seule la visualisation — et la transparence du composé — autorise.

## Liens

- [MGS-1 Introduction](MGS-1-Introduction.ipynb) -- le moteur autonome `MetaGeneticAlgorithm`
- [MGS-2 Composition](MGS-2-Composition.ipynb) -- primitives `Match` et grammaire fluente
- [MGS-3 Eukaryote](MGS-3-Eukaryote.ipynb) -- sous-populations spécialisées
- [MGS-4 Islands](MGS-4-Islands.ipynb) -- modèle insulaire et migration
- [MGS-5 Benchmarks](MGS-5-Benchmarks.ipynb) -- banc comparatif (ce notebook en explique visuellement les verdicts)
- [MGS-6 TSP](MGS-6-TSP.ipynb) -- composition agnostique à la représentation
- [Point d'entrée Part 4](../Part4-Metaheuristics/README.md) -- positionnement MGS vs GeneticSharp/mealpy/HeuristicLab
- [Fork jsboige/MetaGeneticSharp](https://github.com/jsboige/MetaGeneticSharp) -- code source, `KnownFunctions.cs`, contrôleur gtk# original
